Анализ корпоративных данных и факторов релевантности новостей

Этот ноутбук загружает и проверяет CSV-файлы из папки `data/`, рассчитывает показатели компаний, групп, связей и платежей.


In [29]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"

companies = pd.read_csv(DATA / "df_Uid_data_hack.csv", sep=";", encoding="utf-8-sig", encoding_errors="replace")
links = pd.read_csv(DATA / "df_Fid_Uid_link_hack.csv", sep=";", encoding="utf-8-sig", encoding_errors="replace")
payments = pd.read_csv(DATA / "df_Uid_payment_hack.csv", sep=";", encoding="cp1251")

print(f"ROOT: {ROOT.resolve()}")
print(f"companies: {companies.shape}; links: {links.shape}; payments: {payments.shape}")
print("company columns:", list(companies.columns))
print("link columns:", list(links.columns))
print("payment columns:", list(payments.columns))


ROOT: C:\Users\Asus\Desktop\alfa-hackathon-2026
companies: (85525, 24); links: (12514, 4); payments: (1380585, 4)
company columns: ['SYNTHETIC_U', 'ORG_TYPE', 'OKFS_NAME', 'COMPANY_AGE_YEARS_GRADATION', 'ECO_MARK_LABEL', 'MAINOKVED_CODE', 'MAINOKVED_NAME', 'NEGATIVEMARK', 'AMOUNT_WORKERS_FNS_GRADATION', 'AMOUNT_WORKERS_SPARK_GRADATION', 'CL_TYPE', 'CL_PRODUCT_RKO_MARK', 'CL_PRODUCT_GAR_MARK', 'CL_PRODUCT_LOA_MARK', 'CL_PRODUCT_LEA_MARK', 'CL_PRODUCT_DEP_MARK', 'CL_PRODUCT_ZPP_MARK', 'FTS_FLAG', 'FTS_COUNTRY_NAME', 'FACTORING_MARK', 'REVENUE_COMPANY', 'GROUP_ID', 'CITY', 'REGION']
link columns: ['SYNTHETIC_F', 'CITY_NAME', 'RC_NAME', 'SYNTHETIC_U']
payment columns: ['BENEFICIARY', 'PAYER', 'PAY_AMT', 'PAY_CNT']


## 2. Проверка схемы и качества данных

`SYNTHETIC_U` рассматривается как кандидат на ключ, но его уникальность проверяется до объединения таблиц.


In [22]:
expected_company = {
    "SYNTHETIC_U", "ORG_TYPE", "OKFS_NAME", "COMPANY_AGE_YEARS_GRADATION",
    "ECO_MARK_LABEL", "MAINOKVED_CODE", "MAINOKVED_NAME", "NEGATIVEMARK",
    "AMOUNT_WORKERS_FNS_GRADATION", "AMOUNT_WORKERS_SPARK_GRADATION", "CL_TYPE",
    "CL_PRODUCT_RKO_MARK", "CL_PRODUCT_GAR_MARK", "CL_PRODUCT_LOA_MARK",
    "CL_PRODUCT_LEA_MARK", "CL_PRODUCT_DEP_MARK", "CL_PRODUCT_ZPP_MARK",
    "FTS_FLAG", "FTS_COUNTRY_NAME", "FACTORING_MARK", "REVENUE_COMPANY",
    "GROUP_ID", "CITY", "REGION"
}
expected_link = {"SYNTHETIC_F", "CITY_NAME", "RC_NAME", "SYNTHETIC_U"}
expected_payment = {"BENEFICIARY", "PAYER", "PAY_AMT", "PAY_CNT"}

quality = pd.DataFrame({
    "rows": [len(companies), len(links), len(payments)],
    "columns": [companies.shape[1], links.shape[1], payments.shape[1]],
    "duplicate_rows": [companies.duplicated().sum(), links.duplicated().sum(), payments.duplicated().sum()],
}, index=["companies", "links", "payments"])
display(quality)

print("company unique SYNTHETIC_U:", companies["SYNTHETIC_U"].nunique())
print("duplicate company keys:", companies["SYNTHETIC_U"].duplicated().sum())
print("missing values (%):")
print(companies.isnull().sum())
print(links.isnull().sum())
print(payments.isnull().sum())
display((companies.isna().mean().mul(100).sort_values(ascending=False).head(8).round(2)).to_frame("missing_pct"))

,rows,columns,duplicate_rows
companies,85525,24,0
links,12514,4,0
payments,1380585,4,1


company unique SYNTHETIC_U: 85514
duplicate company keys: 11
missing values (%):
SYNTHETIC_U                           0
ORG_TYPE                              0
OKFS_NAME                             0
COMPANY_AGE_YEARS_GRADATION           0
ECO_MARK_LABEL                     4157
MAINOKVED_CODE                        0
MAINOKVED_NAME                        0
NEGATIVEMARK                          0
AMOUNT_WORKERS_FNS_GRADATION          0
AMOUNT_WORKERS_SPARK_GRADATION    85357
CL_TYPE                               0
CL_PRODUCT_RKO_MARK                   0
CL_PRODUCT_GAR_MARK                   0
CL_PRODUCT_LOA_MARK                   0
CL_PRODUCT_LEA_MARK                   0
CL_PRODUCT_DEP_MARK                   0
CL_PRODUCT_ZPP_MARK                   0
FTS_FLAG                              0
FTS_COUNTRY_NAME                  77179
FACTORING_MARK                        0
REVENUE_COMPANY                       0
GROUP_ID                              0
CITY                                  0

,missing_pct
AMOUNT_WORKERS_SPARK_GRADATION,99.80
FTS_COUNTRY_NAME,90.24
ECO_MARK_LABEL,4.86
SYNTHETIC_U,0.00
COMPANY_AGE_YEARS_GRADATION,0.00
MAINOKVED_CODE,0.00
OKFS_NAME,0.00
ORG_TYPE,0.00


In [23]:
company_ids = set(companies.SYNTHETIC_U)
link_coverage = links.SYNTHETIC_U.isin(company_ids).mean()
payer_coverage = payments.PAYER.isin(company_ids).mean()
beneficiary_coverage = payments.BENEFICIARY.isin(company_ids).mean()

coverage = pd.Series({
    "link company id coverage": link_coverage,
    "payer id coverage": payer_coverage,
    "beneficiary id coverage": beneficiary_coverage,
})
display((coverage * 100).round(2).to_frame("coverage_pct"))

print("groups:", companies.GROUP_ID.nunique(), "missing group ids:", companies.GROUP_ID.isna().sum())
print("industries:", companies.MAINOKVED_CODE.nunique(), "cities:", companies.CITY.nunique(), "regions:", companies.REGION.nunique())


,coverage_pct
link company id coverage,36.99
payer id coverage,57.83
beneficiary id coverage,70.37


groups: 18098 missing group ids: 0
industries: 546 cities: 267 regions: 82


## 3. Профиль компаний и корпоративных групп

Выручка группы ниже рассчитана как сумма наблюдаемых значений отдельных компаний. Это не консолидированная выручка.


In [24]:
revenue_stats = companies.REVENUE_COMPANY.describe(percentiles=[.25, .5, .75, .9, .99]).round(2)
display(revenue_stats.to_frame("REVENUE_COMPANY"))

profile = {
    "customer_status": companies.CL_TYPE.value_counts(dropna=False).to_dict(),
    "age_category": companies.COMPANY_AGE_YEARS_GRADATION.value_counts(dropna=False).to_dict(),
    "workers_fts": companies.AMOUNT_WORKERS_FNS_GRADATION.value_counts(dropna=False).to_dict(),
    "fts_country_non_null": int(companies.FTS_COUNTRY_NAME.notna().sum()),
}
print(profile)

products = ["CL_PRODUCT_RKO_MARK", "CL_PRODUCT_GAR_MARK", "CL_PRODUCT_LOA_MARK",
            "CL_PRODUCT_LEA_MARK", "CL_PRODUCT_DEP_MARK", "CL_PRODUCT_ZPP_MARK", "FACTORING_MARK"]
product_rates = (companies[products].eq(1).mean().mul(100).sort_values(ascending=False).round(2))
display(product_rates.to_frame("adoption_pct"))

customer_revenue = companies.groupby("CL_TYPE", dropna=False).agg(
    companies=("SYNTHETIC_U", "nunique"),
    median_revenue=("REVENUE_COMPANY", "median"),
    observed_revenue=("REVENUE_COMPANY", "sum"),
).sort_values("median_revenue", ascending=False)
display(customer_revenue)


,REVENUE_COMPANY
count,8.552500e+04
mean,3.797200e+08
std,1.425250e+09
min,1.000000e+03
25%,8.835000e+05
50%,3.358343e+07
75%,2.401415e+08
90%,9.097603e+08
99%,5.270933e+09
max,1.055965e+11


{'customer_status': {'CLIENT': 41968, 'PROSPECT': 36699, 'CLOSE_CLIENT': 6858}, 'age_category': {'8-15': 33103, '16-25': 21288, '0-3': 16250, '4-7': 14884}, 'workers_fts': {'Менее 15': 59111, '16-100': 23264, '101-250': 2380, 'Более 251': 770}, 'fts_country_non_null': 8346}


,adoption_pct
CL_PRODUCT_RKO_MARK,46.38
CL_PRODUCT_ZPP_MARK,30.97
CL_PRODUCT_LEA_MARK,17.55
CL_PRODUCT_LOA_MARK,5.78
CL_PRODUCT_DEP_MARK,5.01
CL_PRODUCT_GAR_MARK,3.26
FACTORING_MARK,0.25


,companies,median_revenue,observed_revenue
CL_TYPE,,,
CLIENT,41968,72626468.0,2.203461e+13
CLOSE_CLIENT,6858,57678621.5,2.983328e+12
PROSPECT,36688,10515281.0,7.457620e+12


In [25]:
group_summary = companies.groupby("GROUP_ID", dropna=False).agg(
    companies=("SYNTHETIC_U", "nunique"),
    observed_revenue=("REVENUE_COMPANY", "sum"),
    average_revenue=("REVENUE_COMPANY", "mean"),
    industries=("MAINOKVED_CODE", "nunique"),
    regions=("REGION", "nunique"),
).sort_values("observed_revenue", ascending=False)
print("group size distribution:")
display(group_summary["companies"].describe().to_frame("companies_per_group"))
print("singleton groups:", int((group_summary.companies == 1).sum()))
display(group_summary.head(10).round(2))


group size distribution:


,companies_per_group
count,18098.000000
mean,4.725605
std,21.461634
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,624.000000


singleton groups: 12869


,companies,observed_revenue,average_revenue,industries,regions
GROUP_ID,,,,,
group_134402_sb,369,3.027260e+11,8.203958e+08,114,31
group_134409_sb,482,2.194658e+11,4.553233e+08,109,12
group_134761_sb,109,1.368770e+11,1.255752e+09,27,6
group_134457_sb,260,1.210975e+11,4.657597e+08,87,15
group_134498_sb,157,1.137823e+11,7.247280e+08,72,16
group_134460_sb,215,1.102589e+11,5.128322e+08,78,17
group_134404_sb,624,1.017392e+11,1.630435e+08,135,29
group_134545_sb,131,9.465084e+10,7.225255e+08,58,7
group_134407_sb,441,9.448036e+10,2.142412e+08,121,16


## 4. Связи физических лиц и платежи

Общее анонимизированное лицо является наблюдаемой связью, но не доказательством владения или контроля. Плательник отправляет средства, а получатель их получает.


In [26]:
company_people = links.groupby("SYNTHETIC_U").agg(
    distinct_people=("SYNTHETIC_F", "nunique"),
    relationship_rows=("SYNTHETIC_F", "size"),
)
people_companies = links.groupby("SYNTHETIC_F").agg(
    distinct_companies=("SYNTHETIC_U", "nunique"),
)
print("companies with observed links:", len(company_people))
print("people linked to >1 company:", int((people_companies.distinct_companies > 1).sum()))
print("maximum companies per person:", int(people_companies.distinct_companies.max()))

unique_links = links[["SYNTHETIC_F", "SYNTHETIC_U"]].drop_duplicates()
pairs = unique_links.merge(unique_links, on="SYNTHETIC_F")
pairs = pairs[pairs.SYNTHETIC_U_x < pairs.SYNTHETIC_U_y][["SYNTHETIC_U_x", "SYNTHETIC_U_y"]].drop_duplicates()
print("company pairs sharing a person:", len(pairs))


companies with observed links: 12307
people linked to >1 company: 969
maximum companies per person: 25
company pairs sharing a person: 2779


In [27]:
payment_duplicate_rows = payments[payments.duplicated(keep=False)]
print("duplicate payment rows:")
display(payment_duplicate_rows)
print("payment amount stats:")
display(payments.PAY_AMT.describe(percentiles=[.5, .9, .99]).round(2).to_frame("PAY_AMT"))
print("payment count stats:")
display(payments.PAY_CNT.describe(percentiles=[.5, .9, .99]).round(2).to_frame("PAY_CNT"))

self_transfer = payments.BENEFICIARY.eq(payments.PAYER)
print("self-transfer rows:", int(self_transfer.sum()))
print("self-transfer amount share:", round(payments.loc[self_transfer, "PAY_AMT"].sum() / payments.PAY_AMT.sum() * 100, 2), "%")

outgoing = payments.groupby("PAYER").agg(
    outgoing_volume=("PAY_AMT", "sum"),
    outgoing_payment_count=("PAY_CNT", "sum"),
    outgoing_counterparties=("BENEFICIARY", "nunique"),
).sort_values("outgoing_volume", ascending=False)
incoming = payments.groupby("BENEFICIARY").agg(
    incoming_volume=("PAY_AMT", "sum"),
    incoming_payment_count=("PAY_CNT", "sum"),
    incoming_counterparties=("PAYER", "nunique"),
).sort_values("incoming_volume", ascending=False)
print("top outgoing:")
display(outgoing.head(10).round(2))
print("top incoming:")
display(incoming.head(10).round(2))


duplicate payment rows:


,BENEFICIARY,PAYER,PAY_AMT,PAY_CNT
632617,U_84713,U_428697,8000.0,1
1283903,U_84713,U_428697,8000.0,1


payment amount stats:


,PAY_AMT
count,1.380585e+06
mean,4.055617e+07
std,1.814375e+09
min,1.000000e+03
50%,2.120000e+05
90%,8.561000e+06
99%,2.346751e+08
max,1.118581e+12


payment count stats:


,PAY_CNT
count,1380585.00
mean,17.76
std,136.31
min,1.00
50%,5.00
90%,32.00
99%,219.00
max,124671.00


self-transfer rows: 27406
self-transfer amount share: 81.66 %
top outgoing:


,outgoing_volume,outgoing_payment_count,outgoing_counterparties
PAYER,,,
U_33404,1.123791e+12,2590,138
U_75345,8.170168e+11,1522,24
U_179901,5.175065e+11,2250,22
U_98379,3.160014e+11,1164,10
U_86785,3.135778e+11,2053,52
U_31747,3.089549e+11,984,44
U_20228,2.758695e+11,82530,540
U_28927,2.659111e+11,5540,298
U_10578,2.380812e+11,1264,42


top incoming:


,incoming_volume,incoming_payment_count,incoming_counterparties
BENEFICIARY,,,
U_33404,1.120764e+12,1259,24
U_75345,8.266978e+11,4149,43
U_179901,5.172970e+11,1909,21
U_31747,3.160877e+11,3327,149
U_98379,3.154192e+11,1086,5
U_86785,2.939458e+11,398,6
U_28927,2.660294e+11,2436,194
U_20228,2.657287e+11,2677,86
U_10578,2.381008e+11,886,10


In [28]:
group_by_company = companies.set_index("SYNTHETIC_U")["GROUP_ID"].to_dict()
payment_groups = payments.assign(
    payer_group=payments.PAYER.map(group_by_company),
    beneficiary_group=payments.BENEFICIARY.map(group_by_company),
)
known_groups = payment_groups.dropna(subset=["payer_group", "beneficiary_group"])
internal = known_groups.payer_group.eq(known_groups.beneficiary_group)
print("payment rows with both groups known:", len(known_groups), f"({len(known_groups)/len(payments):.2%})")
print("internal-group amount share among known-group rows:", round(known_groups.loc[internal, "PAY_AMT"].sum() / known_groups.PAY_AMT.sum() * 100, 2), "%")


payment rows with both groups known: 389396 (28.21%)
internal-group amount share among known-group rows: 96.38 %
